# Descripción de proyecto

Trabajas como analista de datos para la empresa de telecomunicaciones Megaline, la cual ofrece a sus clientes dos planes de prepago: Surf y Ultimate. El departamento comercial necesita identificar cuál de estos planes genera mayores ingresos, con el fin de optimizar la asignación del presupuesto de publicidad.

Para apoyar esta decisión, se realizará un análisis preliminar de las tarifas utilizando una muestra de clientes. El conjunto de datos incluye información de 500 usuarios de Megaline, como su ubicación, el plan que utilizan y sus patrones de consumo durante el año 2018, incluyendo la cantidad de llamadas realizadas, mensajes de texto enviados y uso de datos móviles.

Además de estimar los ingresos generados por cada plan, el análisis también se enfocará en comprender el comportamiento de los clientes, identificando patrones de uso y diferencias en el consumo entre los usuarios de cada tarifa.

El objetivo principal de este análisis es evaluar cómo se comportan los clientes según el plan contratado y determinar cuál de los planes de prepago genera mayores ingresos en promedio. Esta diferencia en ingresos será evaluada posteriormente mediante pruebas estadísticas.

# Descripción de las tarifas
Nota: Megaline redondea los segundos a minutos y los megabytes a gigabytes. Para las llamadas, cada llamada individual se redondea: incluso si la llamada duró solo un segundo, se contará como un minuto. Para el tráfico web, las sesiones web individuales no se redondean. En vez de esto, el total del mes se redondea hacia arriba. Si alguien usa 1025 megabytes este mes, se le cobrarán 2 gigabytes.

A continuación puedes ver una descripción de las tarifas:
    
Surf

Pago mensual: $20.00\
500 minutos al mes, 50 SMS y 15 GB de datos\
Si se exceden los límites del paquete:\
1 minuto: 3 centavos\
1 SMS: 3 centavos\
1 GB de datos: $10.\
Ultimate

Pago mensual: $70.\
3000 minutos al mes, 1000 SMS y 30 GB de datos.\
Si se exceden los límites del paquete:\
1 minuto: 1 centavo\
1 SMS: 1 centavo\
1 GB de datos: $7.00

### Análisis general de los datos
1- Obtener todas las bases de datos y librerías necesarias

2- Visualizar el tipo de datos que se van a manejar

3- Visualizar posibles duplicados 

4- Corregir datos

5- Enriquecer los datos

In [2]:
import pandas as pd
from scipy import stats as st
import numpy as np
import matplotlib.pyplot as plt
import math
# -----------------------------------------------------------------------#
meg_calls = pd.read_csv('datasets/megaline_calls.csv')
meg_internet = pd.read_csv('datasets/megaline_internet.csv')
meg_msm = pd.read_csv('datasets/megaline_messages.csv')
meg_plans = pd.read_csv('datasets/megaline_plans.csv')
meg_users = pd.read_csv('datasets/megaline_users.csv')

In [3]:
meg_calls.info()
meg_calls.sample(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 137735 entries, 0 to 137734
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   id         137735 non-null  object 
 1   user_id    137735 non-null  int64  
 2   call_date  137735 non-null  object 
 3   duration   137735 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 4.2+ MB


,id,user_id,call_date,duration
65925,1239_8,1239,2018-12-01,4.63
89684,1326_335,1326,2018-09-12,0.00
60954,1220_336,1220,2018-10-26,5.54
71582,1255_64,1255,2018-10-26,0.84
17029,1066_131,1066,2018-08-22,9.83
59384,1214_453,1214,2018-11-19,4.51
92424,1332_80,1332,2018-09-02,0.00
83328,1302_102,1302,2018-10-08,6.81
116434,1408_83,1408,2018-07-26,0.47
1590,1009_12,1009,2018-10-31,13.66


In [4]:
meg_internet.info()
meg_internet.sample(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104825 entries, 0 to 104824
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            104825 non-null  object 
 1   user_id       104825 non-null  int64  
 2   session_date  104825 non-null  object 
 3   mb_used       104825 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 3.2+ MB


,id,user_id,session_date,mb_used
99869,1472_333,1472,2018-04-25,0.00
19088,1085_162,1085,2018-12-09,189.99
74577,1351_63,1351,2018-12-15,828.69
104209,1498_274,1498,2018-08-25,449.00
47350,1211_227,1211,2018-09-07,104.29
93654,1437_12,1437,2018-11-16,313.92
1224,1007_208,1007,2018-11-03,607.81
33531,1152_159,1152,2018-12-31,336.82
100652,1476_169,1476,2018-07-31,203.57
56867,1257_134,1257,2018-05-29,0.00


In [5]:
meg_msm.info()
meg_msm.sample(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76051 entries, 0 to 76050
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            76051 non-null  object
 1   user_id       76051 non-null  int64 
 2   message_date  76051 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.7+ MB


,id,user_id,message_date
42851,1281_42,1281,2018-08-23
22700,1136_78,1136,2018-11-25
1618,1016_126,1016,2018-12-01
14415,1098_453,1098,2018-11-14
20338,1130_13,1130,2018-11-09
71420,1466_72,1466,2018-07-04
56631,1358_19,1358,2018-05-26
16224,1110_115,1110,2018-09-01
44109,1293_858,1293,2018-08-19
43261,1285_93,1285,2018-09-27


In [6]:
meg_plans.info()
meg_plans.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   messages_included      2 non-null      int64  
 1   mb_per_month_included  2 non-null      int64  
 2   minutes_included       2 non-null      int64  
 3   usd_monthly_pay        2 non-null      int64  
 4   usd_per_gb             2 non-null      int64  
 5   usd_per_message        2 non-null      float64
 6   usd_per_minute         2 non-null      float64
 7   plan_name              2 non-null      object 
dtypes: float64(2), int64(5), object(1)
memory usage: 260.0+ bytes


,messages_included,mb_per_month_included,minutes_included,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute,plan_name
0,50,15360,500,20,10,0.03,0.03,surf
1,1000,30720,3000,70,7,0.01,0.01,ultimate


In [7]:
meg_users.info()
meg_users.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     500 non-null    int64 
 1   first_name  500 non-null    object
 2   last_name   500 non-null    object
 3   age         500 non-null    int64 
 4   city        500 non-null    object
 5   reg_date    500 non-null    object
 6   plan        500 non-null    object
 7   churn_date  34 non-null     object
dtypes: int64(2), object(6)
memory usage: 31.4+ KB


,user_id,first_name,last_name,age,city,reg_date,plan,churn_date
246,1246,Stevie,Moran,69,"Tampa-St. Petersburg-Clearwater, FL MSA",2018-01-09,ultimate,2018-07-31
322,1322,Tyler,Sweeney,22,"Fresno, CA MSA",2018-06-18,surf,NaN
139,1139,Thomas,Lawson,66,"New Orleans-Metairie, LA MSA",2018-11-18,surf,NaN
339,1339,Mariette,Mclean,27,"Charlotte-Concord-Gastonia, NC-SC MSA",2018-03-03,surf,NaN
373,1373,Lindsey,Dickerson,21,"Minneapolis-St. Paul-Bloomington, MN-WI MSA",2018-08-30,ultimate,NaN


Todos estos datasets tienen distintos propositos, como los 3 primeros que tiene informacion acerca del consumo de cada servicio proporcionado, no para internet, otro para mensajes y etc.
Uno con infomacion acerca de los usuarios y otro acerca de los datos de las tarifas que ofrece esta empresa.

Se puede ver que en todos los datasets el tipo de dato para las fechas es incorrecto.

En "churn date" hay información vacía, sin embargo en este caso se puede conservar este estado ya que indica que no se han cancelado las suscripciones.

En la base de datos meg_plans, los nombres de las columnas son demasiado largas, esto se puede reducir.

aqui se vio algo interesante y es que en varias ocaciones aparece el comsumo de internet en 0 o llamdas con duracion de 0, esto puede significar dos cosas que se estan tomando todos los dias sin excepcion o que se recopilo esta informacion por error

Además, en base a las condiciones de la suscripción se debe de redondear hacia arriba el internet consumido, así que esto se debe de corregir.

# Comprobacion de informacion

In [8]:
comprobacion = meg_internet.sort_values(
    by=['user_id', 'session_date'], ascending=True)
usuario = np.random.choice(meg_users['user_id'])
print(comprobacion[(comprobacion)['user_id'] == usuario])

             id  user_id session_date  mb_used
65186   1300_53     1300   2018-10-22   435.13
65201   1300_79     1300   2018-10-22     0.00
65253  1300_167     1300   2018-10-22   192.51
65227  1300_124     1300   2018-10-23   437.91
65237  1300_140     1300   2018-10-23   332.86
...         ...      ...          ...      ...
65204   1300_83     1300   2018-12-30   262.13
65233  1300_133     1300   2018-12-30   527.15
65239  1300_142     1300   2018-12-30   598.24
65191   1300_64     1300   2018-12-31     0.00
65208   1300_90     1300   2018-12-31   179.93

[151 rows x 4 columns]


Aqui lo que se hizo es agarrar a un usuario al alzar y mostrar todo su historial de consumo, y demostro que incluso en un mismo dia hay registros con 0 de consumo lo cual se puede explicar si un usuario inicio sesion pero no descargo nada... pasara lo mismo con las llamadas?

In [9]:
comprobacion = meg_calls.sort_values(
    by=['user_id', 'call_date'], ascending=True)
usuario = np.random.choice(meg_users['user_id'])
print(comprobacion[(comprobacion)['user_id'] == usuario])

             id  user_id   call_date  duration
35902  1135_179     1135  2018-12-24      0.00
35903  1135_213     1135  2018-12-24     17.76
35908  1135_314     1135  2018-12-24      1.07
35900   1135_71     1135  2018-12-25      0.35
35906  1135_285     1135  2018-12-25     16.08
35914  1135_428     1135  2018-12-25      5.36
35898    1135_9     1135  2018-12-26      4.88
35899   1135_60     1135  2018-12-27     11.33
35904  1135_222     1135  2018-12-27      4.38
35907  1135_306     1135  2018-12-28      0.00
35910  1135_383     1135  2018-12-28     19.17
35905  1135_223     1135  2018-12-29     10.63
35909  1135_334     1135  2018-12-29     11.80
35912  1135_413     1135  2018-12-29      9.13
35901   1135_72     1135  2018-12-30      8.55
35911  1135_398     1135  2018-12-30     11.50
35913  1135_422     1135  2018-12-30      1.81


Demuestra el mismo comportamiento teniendo como explicacion que el conteo de llamdas se cuenta desde el momento que contesta la otra persona, los registros en 0 puede significar que esta llamada no fue contestada.

Dado que nuestro objetivo de investigacion se centra principalmente en analizar el consumo realizadio y las ganancias generadas de cada plan esta informacion no es relevante, no se eliminara pero tampoco sera tomada en cuenta

asi que continuaremos a la correcion de los datos de los datasets presentados

# Correccion de errores en los datasets

In [10]:
# Cambio al tipo fecha de las columnas marcadas#

meg_users[['churn_date', 'reg_date']] = meg_users[[
    'churn_date', 'reg_date']].apply(pd.to_datetime)
meg_calls['call_date'] = pd.to_datetime(meg_calls['call_date'])
meg_internet['session_date'] = pd.to_datetime(meg_internet['session_date'])
meg_msm['message_date'] = pd.to_datetime(meg_msm['message_date'])


# Eliminar los datos duplicados#

meg_internet.drop_duplicates(inplace=True)
meg_calls.drop_duplicates(inplace=True)
meg_users.drop_duplicates(inplace=True)
meg_msm.drop_duplicates(inplace=True)

# Renombrar algunas columnas#

meg_plans = meg_plans.rename(columns={'plan_name': 'plan'})

In [11]:
meg_calls.info()
meg_calls.sample(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 137735 entries, 0 to 137734
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   id         137735 non-null  object        
 1   user_id    137735 non-null  int64         
 2   call_date  137735 non-null  datetime64[ns]
 3   duration   137735 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 4.2+ MB


,id,user_id,call_date,duration
114706,1401_564,1401,2018-10-07,0.00
67981,1246_0,1246,2018-08-24,9.03
91668,1329_880,1329,2018-11-07,8.11
10347,1046_213,1046,2018-08-11,2.54
104743,1368_461,1368,2018-05-21,5.15
91001,1328_631,1328,2018-09-18,0.00
80837,1291_437,1291,2018-10-04,0.00
46665,1170_153,1170,2018-11-21,3.19
57993,1209_511,1209,2018-08-31,10.44
101061,1361_163,1361,2018-12-03,10.20


In [12]:
meg_internet.info()
meg_internet.sample(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104825 entries, 0 to 104824
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   id            104825 non-null  object        
 1   user_id       104825 non-null  int64         
 2   session_date  104825 non-null  datetime64[ns]
 3   mb_used       104825 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 3.2+ MB


,id,user_id,session_date,mb_used
66538,1311_185,1311,2018-11-10,301.26
20897,1097_135,1097,2018-08-31,338.16
2430,1011_170,1011,2018-09-12,24.17
81363,1381_42,1381,2018-09-19,287.88
10547,1054_80,1054,2018-12-13,489.55
47187,1211_64,1211,2018-09-20,156.55
64129,1294_50,1294,2018-10-20,275.77
65584,1302_130,1302,2018-11-23,213.55
28723,1132_111,1132,2018-10-18,916.65
96784,1456_9,1456,2018-08-15,738.69


In [13]:
meg_msm.info()
meg_msm.sample(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76051 entries, 0 to 76050
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id            76051 non-null  object        
 1   user_id       76051 non-null  int64         
 2   message_date  76051 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 1.7+ MB


,id,user_id,message_date
33289,1209_140,1209,2018-11-26
68959,1447_27,1447,2018-11-04
33851,1213_54,1213,2018-12-01
62971,1399_36,1399,2018-12-10
42115,1272_10,1272,2018-12-18
4644,1043_1070,1043,2018-10-27
63889,1408_14,1408,2018-07-30
15914,1105_113,1105,2018-10-31
58297,1369_98,1369,2018-12-12
47431,1324_145,1324,2018-09-29


In [14]:
meg_plans.info()
meg_plans.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   messages_included      2 non-null      int64  
 1   mb_per_month_included  2 non-null      int64  
 2   minutes_included       2 non-null      int64  
 3   usd_monthly_pay        2 non-null      int64  
 4   usd_per_gb             2 non-null      int64  
 5   usd_per_message        2 non-null      float64
 6   usd_per_minute         2 non-null      float64
 7   plan                   2 non-null      object 
dtypes: float64(2), int64(5), object(1)
memory usage: 260.0+ bytes


,messages_included,mb_per_month_included,minutes_included,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute,plan
0,50,15360,500,20,10,0.03,0.03,surf
1,1000,30720,3000,70,7,0.01,0.01,ultimate


In [15]:
meg_users.info()
meg_users.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     500 non-null    int64         
 1   first_name  500 non-null    object        
 2   last_name   500 non-null    object        
 3   age         500 non-null    int64         
 4   city        500 non-null    object        
 5   reg_date    500 non-null    datetime64[ns]
 6   plan        500 non-null    object        
 7   churn_date  34 non-null     datetime64[ns]
dtypes: datetime64[ns](2), int64(2), object(4)
memory usage: 31.4+ KB


,user_id,first_name,last_name,age,city,reg_date,plan,churn_date
217,1217,Ned,Thomas,69,"Dayton-Kettering, OH MSA",2018-06-04,surf,NaT
461,1461,Rupert,Santana,54,"Pittsburgh, PA MSA",2018-09-14,surf,NaT
3,1003,Reynaldo,Jenkins,52,"Tulsa, OK MSA",2018-01-28,surf,NaT
98,1098,Collin,Sims,33,"Albany-Schenectady-Troy, NY MSA",2018-08-14,surf,NaT
312,1312,Kory,Emerson,42,"Fresno, CA MSA",2018-01-26,surf,NaT


Con esto ademas de resolver las dudas de los valores en 0, tenemos los datsaets limpios por lo que podemos seguir con en analisis y en este caso con el enriquecimiento de datos ya que nos sera util.

# Enriquecimiento de datos

Se construirán datasets adicionales con el objetivo de organizar la información de manera más clara y facilitar el análisis, así como la generación de conclusiones sobre el comportamiento de los usuarios.

Para ello, se extrajo el componente mensual de las variables de fecha en los distintos datasets, con el fin de estandarizar la dimensión temporal y permitir una agregación consistente de la actividad por ciclo de facturación.

Asimismo, el límite de datos incluidos fue convertido de megabytes a gigabytes, aplicando redondeo hacia arriba (ceil). Esta transformación se realizó para alinearse con las reglas de facturación del negocio y garantizar precisión en el cálculo de ingresos y excedentes de consumo.

In [16]:
meg_msm["period"] = meg_msm["message_date"].dt.month
meg_calls["period"] = meg_calls["call_date"].dt.month
meg_users['month_reg'] = meg_users['reg_date'].dt.month
meg_users['month_churn'] = meg_users['churn_date'].dt.month
meg_internet["period"] = meg_internet["session_date"].dt.month
meg_plans['mb_per_month_included'] = np.ceil(
    meg_plans['mb_per_month_included'] / 1024)

Con las fechas modificadas podemos generar un nuevo dataset donde todas las tablas anteriores se unan en una tabla general

In [ ]:
# Agregar los datos por usuario y por periodo de cada dataset#
llamadas_mes = meg_calls.groupby(['user_id', 'period']).size().reset_index()
min_llamadas = meg_calls.groupby(['user_id', 'period'])[
    'duration'].sum().reset_index()
n_mensajes = meg_msm.groupby(['user_id', 'period']).size().reset_index()
inter_trafic = meg_internet.groupby(['user_id', 'period'])[
    'mb_used'].sum().reset_index()

# Renombre de columnas#

llamadas_mes = llamadas_mes.rename(columns={0: 'calls_n'})
n_mensajes.rename(columns={0: 'n_msm'}, inplace=True)

# Creacion de un nuevo dataset
# #
total = pd.merge(llamadas_mes, min_llamadas, on=[
                 'user_id', 'period'], how='outer')
total = pd.merge(total, inter_trafic, on=['user_id', 'period'], how='outer')
total = pd.merge(total, n_mensajes, on=['user_id', 'period'], how='outer')
total = pd.merge(
    total, meg_users[['user_id', 'plan', 'city']], on='user_id', how='outer')
total = total.fillna(0)
total['mb_used'] = np.ceil(total['mb_used'] / 1024)
total['duration'] = np.ceil(total['duration'])
total.info()
total
total.to_excel('datasets/total.xlsx', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2303 entries, 0 to 2302
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   user_id   2303 non-null   int64  
 1   period    2303 non-null   float64
 2   calls_n   2303 non-null   float64
 3   duration  2303 non-null   float64
 4   mb_used   2303 non-null   float64
 5   n_msm     2303 non-null   float64
 6   plan      2303 non-null   object 
 7   city      2303 non-null   object 
dtypes: float64(5), int64(1), object(2)
memory usage: 144.1+ KB


Con el dataset anterior se hará un nuevo dataset en donde se expondrán dos cosas: los límites que uno puede consumir por período y, al lado, lo que ha consumido extra en ese período. Teniendo como columna final lo que el usuario va a pagar en total, este total se calculará teniendo en cuenta la tarifa mencionada al inicio del proyecto y sumando lo extra por cada plan.

In [19]:
# Calculamos los excedentes
total_extra = total.merge(meg_plans, on='plan')
pares = [
    ('duration', 'minutes_included', 'minutos'),
    ('n_msm', 'messages_included', 'mensajes'),
    ('mb_used', 'mb_per_month_included', 'mb')
]
for usado, limite, prefijo in pares:
    total_extra[f'{prefijo}_dentro_plan'] = total_extra[[
        usado, limite]].min(axis=1)
    total_extra[f'{prefijo}_excedido'] = (
        total_extra[usado] - total_extra[limite]).clip(lower=0)
nuevo_orden = [
    'user_id', 'period', 'plan',
    'messages_included', 'mensajes_dentro_plan', 'mensajes_excedido',
    'mb_per_month_included', 'mb_dentro_plan', 'mb_excedido',
    'minutes_included', 'minutos_dentro_plan', 'minutos_excedido',
    'usd_monthly_pay', 'usd_per_gb', 'usd_per_message', 'usd_per_minute'
]
total_extra = total_extra[nuevo_orden]
total_extra['pago_total'] = (total_extra['usd_monthly_pay'] +
                             (total_extra['mb_excedido'] * total_extra['usd_per_gb']) +
                             (total_extra['mensajes_excedido'] * total_extra['usd_per_message']) +
                             (total_extra['minutos_excedido'] * total_extra['usd_per_minute']))
total_extra.info()
total_extra

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2303 entries, 0 to 2302
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                2303 non-null   int64  
 1   period                 2303 non-null   float64
 2   plan                   2303 non-null   object 
 3   messages_included      2303 non-null   int64  
 4   mensajes_dentro_plan   2303 non-null   float64
 5   mensajes_excedido      2303 non-null   float64
 6   mb_per_month_included  2303 non-null   float64
 7   mb_dentro_plan         2303 non-null   float64
 8   mb_excedido            2303 non-null   float64
 9   minutes_included       2303 non-null   int64  
 10  minutos_dentro_plan    2303 non-null   float64
 11  minutos_excedido       2303 non-null   float64
 12  usd_monthly_pay        2303 non-null   int64  
 13  usd_per_gb             2303 non-null   int64  
 14  usd_per_message        2303 non-null   float64
 15  usd_

,user_id,period,plan,messages_included,mensajes_dentro_plan,mensajes_excedido,mb_per_month_included,mb_dentro_plan,mb_excedido,minutes_included,minutos_dentro_plan,minutos_excedido,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute,pago_total
0,1000,12.0,ultimate,1000,11.0,0.0,30.0,2.0,0.0,3000,117.0,0.0,70,7,0.01,0.01,70.00
1,1001,8.0,surf,50,30.0,0.0,15.0,7.0,0.0,500,172.0,0.0,20,10,0.03,0.03,20.00
2,1001,9.0,surf,50,44.0,0.0,15.0,14.0,0.0,500,298.0,0.0,20,10,0.03,0.03,20.00
3,1001,10.0,surf,50,50.0,3.0,15.0,15.0,7.0,500,375.0,0.0,20,10,0.03,0.03,90.09
4,1001,11.0,surf,50,36.0,0.0,15.0,15.0,4.0,500,405.0,0.0,20,10,0.03,0.03,60.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2298,1498,12.0,surf,50,0.0,0.0,15.0,15.0,8.0,500,325.0,0.0,20,10,0.03,0.03,100.00
2299,1499,9.0,surf,50,0.0,0.0,15.0,13.0,0.0,500,331.0,0.0,20,10,0.03,0.03,20.00
2300,1499,10.0,surf,50,0.0,0.0,15.0,15.0,5.0,500,364.0,0.0,20,10,0.03,0.03,70.00
2301,1499,11.0,surf,50,0.0,0.0,15.0,15.0,2.0,500,289.0,0.0,20,10,0.03,0.03,40.00


Y a continuacion se agregara una nueva columna y la informacion que agregar sera la de entregar el pago total que va a efectuar un cliente en terminado periodo por mes

In [20]:
# calculamos los costos
def calcular_ingresos(df):
    # Calcular cargos por excedentes
    costo_minutos = df['minutos_excedido'] * df['usd_per_minute']
    costo_mensajes = df['mensajes_excedido'] * df['usd_per_message']
    costo_datos = df['mb_excedido'] * df['usd_per_gb']
    df['ingreso_mensual'] = df['usd_monthly_pay'] +\
        costo_minutos + costo_mensajes + costo_datos

    return df


df_s = calcular_ingresos(total_extra)
df_s
total = pd.merge(
    total,
    df_s[['user_id', 'period', 'ingreso_mensual']],
    on=['user_id', 'period'],
    how='outer'
)
total.info()
total

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2303 entries, 0 to 2302
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   user_id          2303 non-null   int64  
 1   period           2303 non-null   float64
 2   calls_n          2303 non-null   float64
 3   duration         2303 non-null   float64
 4   mb_used          2303 non-null   float64
 5   n_msm            2303 non-null   float64
 6   plan             2303 non-null   object 
 7   city             2303 non-null   object 
 8   ingreso_mensual  2303 non-null   float64
dtypes: float64(6), int64(1), object(2)
memory usage: 162.1+ KB


,user_id,period,calls_n,duration,mb_used,n_msm,plan,city,ingreso_mensual
0,1000,12.0,16.0,117.0,2.0,11.0,ultimate,"Atlanta-Sandy Springs-Roswell, GA MSA",70.00
1,1001,8.0,27.0,172.0,7.0,30.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",20.00
2,1001,9.0,49.0,298.0,14.0,44.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",20.00
3,1001,10.0,65.0,375.0,22.0,53.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",90.09
4,1001,11.0,64.0,405.0,19.0,36.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",60.00
...,...,...,...,...,...,...,...,...,...
2298,1498,12.0,39.0,325.0,23.0,0.0,surf,"New York-Newark-Jersey City, NY-NJ-PA MSA",100.00
2299,1499,9.0,41.0,331.0,13.0,0.0,surf,"Orlando-Kissimmee-Sanford, FL MSA",20.00
2300,1499,10.0,53.0,364.0,20.0,0.0,surf,"Orlando-Kissimmee-Sanford, FL MSA",70.00
2301,1499,11.0,45.0,289.0,17.0,0.0,surf,"Orlando-Kissimmee-Sanford, FL MSA",40.00


# Estudio del comportamiendo de los clientes